# CANDIDATE FEATURE IDENTIFICATION
The purpose of this notebook is to aggregate in one location the potential features to include in the training set. 
Inputs: External Dataset Parquet Files. 
Desired Outputs: 
- Of redundant variables, select the most reliable and informative one. 
- Understand how to calculate any computed features.

Notes
- Use a separate .py module under src/external_dataset_name/ingest.py to pull in the data with API or other method and write to data/interim/external_dataset_name.parquet. 
- This notebook should read in just a small sample of each dataset for feature exploration. Reading in the full dataset may crash the Jupyter kernel. 

For each feature, attempt to identify:
- time of observation
- unit of observation / what it actually is
- geographic scope (i.e. is it at the census block group, tract, or zip code level?)
- missingness
- what preprocessing is required (e.g. geocoding to census block group, converting a raw count to density)

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
pd.set_option('display.max_columns', None)

## Lottery and Sales Data (for reference when merging)

In [3]:
lottery_locations_file_path = Path(".").resolve().parent / "data/final/location_data.parquet"

#locations_pq = pq.ParquetFile(lottery_locations_file_path)
#locations_first_100_rows = next(locations_pq.iter_batches(batch_size=100))
#locations_df = pa.Table.from_batches([locations_first_100_rows]).to_pandas()

locations_df = pd.read_parquet(str(lottery_locations_file_path))

In [4]:
locations_df.head(2)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid
0,97637,TURCA GROCERY CORP,1142 E 165TH ST,BRONX,10459,-73.88748,40.82523,b'\x01\x01\x00\x00\x00\xa9\xf6\xe9x\xccxR\xc0\...,360050119005,36005011900
1,102843,VALENCIA BAKERY,499 E 138TH ST,BRONX,10454,-73.91962,40.80773,b'\x01\x01\x00\x00\x00\xd7/\xd8\r\xdbzR\xc0\xc...,360050039005,36005003900


In [5]:
lottery_sales_file_path = Path(".").resolve().parent / "data/final/sales_data.parquet"

#sales_pq = pq.ParquetFile(lottery_sales_file_path)
#sales_first_100_rows = next(sales_pq.iter_batches(batch_size=1000))
#sales_df = pa.Table.from_batches([sales_first_100_rows]).to_pandas()

sales_df = pd.read_parquet(str(lottery_sales_file_path))

In [6]:
sales_df.sample(5)

,agtno,total_sales,first_date,last_date,first_pos_date,last_pos_date,pos_sales_days,num_cal_days,num_active_days,days_before_first_sale,days_after_last_sale,avg_cal_daily_sales,avg_active_daily_sales,use_active_period,avg_daily_sales
6976,115698,461436.5,2025-02-26,2025-12-31,2025-02-26,2025-12-31,309.0,309,309.0,0.0,0.0,1493.322006,1493.322006,False,1493.322006
6659,115364,1815933.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,4975.158904,4975.158904,False,4975.158904
7264,116020,24461.5,2025-07-01,2025-12-31,2025-07-02,2025-12-31,171.0,184,183.0,1.0,0.0,132.942935,133.669399,False,132.942935
6256,114917,1359504.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,3724.668493,3724.668493,False,3724.668493
11399,73580,939268.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,364.0,365,365.0,0.0,0.0,2573.338356,2573.338356,False,2573.338356


## Overture Maps Data

In [7]:
overture_file_path = Path(".").resolve().parent / "data/interim/overture_filtered.parquet"

overture_pq = pq.ParquetFile(overture_file_path)
overture_first_100_rows = next(overture_pq.iter_batches(batch_size=100))
overture_df = pa.Table.from_batches([overture_first_100_rows]).to_pandas()

In [8]:
overture_df.sample(2)

,id,geometry,longitude,latitude,basic_category,taxonomy_primary,taxonomy_hierarchy,confidence,brand_name,first_seen
34,10c5c1e8-d783-4242-a978-39dbcdd0f763,b'\x01\x01\x00\x00\x00\xba\xb4+>\xcf<R\xc0\xfd...,-72.950149,40.899312,storage_facility,self_storage_facility,"[services_and_business, storage_facility, self...",0.919912,NaN,2025-06-25.0
97,9e1a1770-5fe9-4ec5-9f90-8b15f743d6a1,"b'\x01\x01\x00\x00\x00*\xca\x84\xe3""9R\xc0\x82...",-72.892754,40.896737,government_office,government_office,"[community_and_government, government_office]",0.919912,NaN,2025-06-25.0


## LODES Data

In [9]:
rac_file_path = Path(".").resolve().parent / "data/final/location_data_rac.parquet"

rac_pq = pq.ParquetFile(rac_file_path)
rac_first_100_rows = next(rac_pq.iter_batches(batch_size=100))
rac_df = pa.Table.from_batches([rac_first_100_rows]).to_pandas()

In [11]:
rac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,rac_c000,rac_ce01,rac_ce02,rac_ce03
73,101637,RALPH AVENUE FOOD CORP,444 RALPH AVE,BROOKLYN,11233,-73.92211,40.67388,"b'\x01\x01\x00\x00\x00""T\xa9\xd9\x03{R\xc0\xd8...",360470303002,36047030300,351.0,48.0,124.0,179.0
3,34228,SAURO'S DELICATESSEN,1072 ROUTE 311,PATTERSON,12563,-73.61114,41.51452,b'\x01\x01\x00\x00\x00\xc2Q\xf2\xea\x1cgR\xc0\...,360790102021,36079010202,419.0,81.0,87.0,251.0


In [12]:
wac_file_path = Path(".").resolve().parent / "data/final/location_data_wac.parquet"

wac_pq = pq.ParquetFile(wac_file_path)
wac_first_100_rows = next(wac_pq.iter_batches(batch_size=100))
wac_df = pa.Table.from_batches([wac_first_100_rows]).to_pandas()

In [13]:
wac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,wac_c000,wac_cns07,wac_cns18,wac_ce01,wac_ce02,wac_ce03
40,68056,7-ELEVEN #16361E,33 MIDDLE COUNTRY RD RTE 25,CORAM,11727,-72.97863,40.87459,b'\x01\x01\x00\x00\x00\x9b8\xb9\xdf\xa1>R\xc0N...,361031583091,36103158309,209.0,36.0,4.0,37.0,68.0,104.0
57,74154,RITE AID #3597,535 PORTLAND AVE,ROCHESTER,14621,-77.59146,43.17565,b'\x01\x01\x00\x00\x00\x199\x0b{\xdaeS\xc0m\xc...,360550080002,36055008000,36.0,1.0,4.0,10.0,16.0,10.0


## AADT Data

In [14]:
aadt_file_path = Path(".").resolve().parent / "data/final/location_data_aadt.parquet"

aadt_pq = pq.ParquetFile(aadt_file_path)
aadt_first_100_rows = next(aadt_pq.iter_batches(batch_size=100))
aadt_df = pa.Table.from_batches([aadt_first_100_rows]).to_pandas()

In [17]:
aadt_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,road_name_norm,municipality_norm,aadt_mean,addt_max,addt_segment_count
79,111381,7-ELEVEN #34350J,6230 JERICHO TPKE UNIT F,COMMACK,11725,-73.29978,40.84190,b'\x01\x01\x00\x00\x00\xac\xffs\x98/SR\xc0O@\x...,361031121031,36103112103,JERICHOTPKEUNITF,COMMACK,<NA>,<NA>,<NA>
60,112604,IN N OUT CONVENIENCE & SMOKE S,343 ROCKAWAY TPKE UNIT 1,LAWRENCE,11559,-73.73817,40.62905,b'\x01\x01\x00\x00\x00\xda8b->oR\xc0?\xc6\xdc\...,360594110001,36059411000,ROCKAWAYTPKEUNIT1,LAWRENCE,<NA>,<NA>,<NA>
7,112949,7-ELEVEN #36951B,179-55 HILLSIDE AVE,JAMAICA,11432,-73.78261,40.71295,b'\x01\x01\x00\x00\x00r\xe1@H\x16rR\xc0s\xd7\x...,360810466002,36081046600,HILLSIDEAVE,JAMAICA,<NA>,<NA>,<NA>
85,111946,STOP N GO,135 BABYLON TPKE,ROOSEVELT,11575,-73.58484,40.67671,b'\x01\x01\x00\x00\x00!Y\xc0\x04neR\xc0&p\xebn...,360594140021,36059414002,BABYLONTPKE,ROOSEVELT,<NA>,<NA>,<NA>
86,113582,OCOA DELI GROCERY INC,1201 LELAND AVE,BRONX,10472,-73.86316,40.83041,b'\x01\x01\x00\x00\x00\xc8\xcdp\x03>wR\xc0\xbc...,360050070003,36005007000,LELANDAVE,BRONX,1738.333333,2307,3
